# Multiprocessing Pool
**Note: Multiprocessing does not work in jupyter notebooks. They need to be run from python files.**

Process pools are a design pattern that let you execute and manage heterogeneus, discrete, and ad hoc tasks.

In [1]:
from time import sleep
from random import random
from multiprocessing import Process, Pool

from math import sqrt, floor

## Processes & Multiprocessing Pools
**1. Process-based concurrency for Full Parallelism**
 
Every Python program is a process and has one thread called the main thread used to execute our program instructions.
 
The `multiprocessing` module and the `Process` class provide process-based concurrency.
 
Process-based concurrency in Python provides true parallelism, that is the ability for a Python program to execute code using more than one CPU core at the same time.

Note: we must explicitly flush the buffer by setting `flush=True` when using the `print()` function from child processes, otherwise the buffer will not flush until the child process terminates.

In [7]:
# custom function to be executed in a child process 
def task():
    # report a message
    print(f"This is another process", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # define a task to run in a new process
    process = Process(target=task)
    # start the task in a new process
    process.start()
    # wait for the child process to terminate
    process.join()

**2. Process pools provide reusable workers**

A process pool is a programming pattern for automatically managing a pool of worker processes. It is responsible for:
* It controls when they are created, such as when they are needed.
* It controls how many tasks each worker can execute before being replaced.
* It also controls what workers should do when they are not being used, such as making them wait without consuming computational resources.

Tasks can be submitted to the pool for execution using synchronous (blocking) and asynchronous (non-blocking) versions of `apply()`, `lambda()`.

In [ ]:
# custom function to be executed in a child process
def task():
    # report a message
    print("This is another process", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue the task
        async_result = pool.apply_async(task)
        # wait for the task to finish
        async_result.wait()
    # close the multiprocessing pool automatially

**3. When to use the multiprocess pool**
* Your tasks can be defined by a pure function that has no state or side effects.
* Your tasks can fit within a single Python function, likely making it simple and easy to understand.
* You need to perform the same task many times with different arguments, e.g., homogeneous tasks.
* You need to apply the same function to each object in a collection in a for-loop.
 
Process pools work best when applying the same pure function on a set of different data, e.g., homogeneous tasks, heterogeneous data.

Use processes for CPU-bound tasks which involve performing a computation and does not involve IO. The operations only involve data in main memory (RAM) or cache (CPU cache) and performing computations on or with that data. As much, the limit of these operations is the speed of the CPU. This is why we call them CPU-bound tasks. Examples include mathematical operations such as calculating the points of a fractal, estimating pi, and factoring primes. It is also appropiate for computational intensive operations such as parsing text documents, encoding images or video and running simulations.

## Configure the Multiprocessing Pools
Arguments:
* `processes`: Maximum number of worker processes to use in the pool.
* `initializer`: Function executed after each worker process is created.
* `initargs`: Arguments to the worker process initialization function.
* `maxtasksperchild`: Limit the maximum number of tasks executed by each worker process.
* `context`: Configure the multiprocessing context such as the process start method.

Default Number Worker Processes = Total Logical CPU Cores in Your System.

Each worker process will be able to execute an unlimited number of tasks within the pool.

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # create a multiprocessing pool
    pool = Pool()
    # report the status of the multiprocessing pool
    print(pool)
    # close the multiprocessing pool
    pool.close()

**1. Configure the Number of Worker Processes**

You should probably set the number of processes to be equal to the number of logical CPU cores in our system e.g., the default.

If we are expecting to perform computational work in the main process in addition to the multiprocessing pool, consider setting the number of processes in the pool to be equal to the number of logical CPUs in our system minus one, to allow the main process to execute.

If we have particularly CPU intensive tasks, consider configuring the number of processes to be equal to the number of physical CPUs instead of the number of logical CPUs.

```python
pool = Pool(processes=4)
```

**2. Configure the Worker Process Initialization**
```python
pool = Pool(initializer=worker_init)
```
If our worker process initialization function takes arguments.
```python
pool = Pool(initializer=init, initargs=(arg1, arg2))
```

In [ ]:
# custom function to be executed in a child process
def task():
    # report a message
    print("Worker executing task...", flush=True)
    # block for a moment
    sleep(1)
    
# initialize a worker in the multiprocessing pool
def initialize_worker():
    # report a message
    print("Initializing worker...", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create and configure the multiprocessing pool
    with Pool(2, initializer=initialize_worker) as pool:
        # issue tasks to the multiprocessing pool
        for _ in range(4):
            pool.apply_async(task)
        # close the multiprocessing pool
        pool.close()
        # wait for all tasks to complete
        pool.join()

**3. Configure the Maximum Tasks per Worker**
```python
pool = Pool(maxtasksperchild=5)
```
The `maxtasksperchild` takes a positive integer number of tasks that may be completed by a child worker process, after which the process will be terminated and a new child worker process will be created to replace it.

Be default the `maxtasksperchild` argument is set to `None`, which means each child worker process will run for the lifetime of the multiprocessing pool.

**4. Configure the Context Used to Create Workers**

The `context` is an instance of a multiprocessing context configured with a start method, created via the `get_context()` function.

By default, `context` is `None`.

A start method is the technique used to start child processes in Python. There are three start methods:
* `spawn`: start a new Python process. Default on Windows and MacOS.
* `fork`: copy a Python process from an existing process. Default for Linux and may not be supported on Windows.
* `forkserver`: New process from which future forked processes will be copied.

```python
# create a process context
ctx = get_context('fork')
# create a multiprocessing pool with a given context
pool = Pool(context=ctx)
```

## Execute Tasks in Parallel and Wait
Blocking call does not return until the call is complete.

Executing tasks asynchronously is helpful when we want to wait for the tasks to complete then process their results.

Blocking method calls for executing tasks on the `Pool` class include:
* `apply()`: For executing one-off tasks.
* `map()`: For calling a function many times with different arguments.
* `starmap()`: For calling a function many times with multiple different arguments.

**1. Run one-off tasks and wait for them to finish.**

`apply()` capabilities:
* Issues a single task to the multiprocessing pool.
* Supports multiple arguments to the target function.
* Block until the call to the target function is complete.
```python
pool.apply(task, args=(arg1, arg2, arg3))
```

In [ ]:
# custom function to be executed in a child process
def task():
    # report a message 
    print("This is another process", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing tool
    with Pool() as pool:
        # issue a task and wait for it to complete
        pool.apply(task)

**2. Run multiple calls to the same function with different arguments, such as in a loop.**

`map()` capabilities:
* Issue multiple tasks to the multiprocessing pool all at once.
* Returns an iterable over return values.
* Supports a single argument to the target function.
* Block until all issued tasks are completed.
* Allows tasks to be grouped and executed in batches by workers.
```python
# iterates return values from the issued tasks
for result in pool.map(task, items, chunksize=10):
    print(result)
```

In [ ]:
# custom function to be executed in a child process
def task(arg):
    # report a message
    print(f"From another process {arg}", flush=True)
    # return a value
    return arg * 2

# protect the entry point 
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue multiple tasks and process return values
        for result in pool.map(task, range(10)):
            print(result)

The tasks complete as workers become available and report a message. Once all tasks are completed, the `map()` method returns an iterable of return values, which is then traversed in the main process.

**3. Run multiple calls to the same function with multiple arguments.**

`starmap()` capabilities:
* Issue multiple tasks to the multiprocessing pool all at once.
* Returns an iterable over return values.
* Supports multiple arguments to the target function.
* Blocks until all issued tasks are completed.
* Allows tasks to be grouped and executed in batches by workers.

```python
# prepare an iterable of iterables for each task
items = [(1, 2), (3, 4), (5, 6)]
# iterates return values from the issued tasks
for result in pool.starmap(task, items, chunksize=10):
    print(result)
```

In [ ]:
# custom function to be executed in a child process
def task(arg1, arg2, arg3):
    # report a message
    print(f"From another process {arg1}, {arg2}, {arg3}", flush=True)
    # return a value
    return arg1 + arg2 + arg3

# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # prepare task arguments
        args = [(i, i*2, i*3) for i in range(10)]
        # issue multiple tasks and process return values
        for result in pool.starmap(task, args):
            print(result)

Tasks are executed as workers become available and report a message with their arguments. Once all tasks are completed, the `starmap()` method returns an iterable of return values, which is then traversed in the main process.

## Execute Tasks in Parallel and No Wait
Execute tasks without blocking. 

Non-blocking call returns immediately, not waiting for the called function to complete and return.

Fire-and-forget or fire-and-monitor

Non-blocking method calls:
* Use `apply_async()`: For executing one-off tasks.
* Use `map_async()`: For calling a function many times with different arguments.
* Use `starmap_async()`: For calling a function many times with multiple different arguments.

**1. Issue one-off tasks asynchronously.**
`apply_async()` capabilities:
* Issues a single task to the multiprocessing pool asynchronously.
* Supports multiple arguments to the target function.
* Does not block, instead returns a `AsyncResult`.
* Supports callback for the return value and any raised errors.
```python
# execute a task in the pool with multiple arguments
async_result = pool.apply_async(task, args=(arg1, arg2))
```

In [ ]:
# custom function to be executed in a child process
def task():
    # report a message
    print(f"This is another process", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue a task asynchronously
        async_result = pool.apply_async(task)
        # wait for the task to complete
        async_result.wait()

**2. Issue multiple calls to the same function with different arguments asynchronously.**
`map_async()` capabilities:
* Issue multiple tasks to the multiprocessing pool all at once asynchronously.
* Supports a single argument to the target function.
* Does not block, instead returns a `AsyncResult` for accessing results later.
* Allows tasks to be grouped and executed in batches by workers.
* Supports callback for the return value and any raised errors.
```python
# issue tasks to the multiprocessing pool asynchronously
async_result = pool.map_async(task, items, chunksize=10)
```

In [ ]:
# custom function to be executed in a child process
def task(arg):
    # report a message
    print(f"From another process {arg}", flush=True)
    # return a value
    return arg * 2

# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue multiple tasks to the pool
        async_result = pool.map_async(task, range(10))
        # process return values once all tasks completed
        for result in async_result.get():
            print(result)

**3. Issue multiple calls to the same function with multiple arguments asynchronously.**

`starmap_async()` capabilities:
* Issue multiple tasks to the multiprocessing pool all at once.
* Supports multiple arguments to the target function.
* Does not block, instead returns a AsyncResult for accessing results later.
* Allows tasks to be grouped and executed in batches by workers.
* Supports callback for the return value and any raised errors.

```python
# prepare an iterable of iterables for each task
items = [(1, 2), (3, 4), (5, 6)]
# issue tasks to the multiprocessing pool asynchronously
async_result = pool.starmap_async(task, items)
```

In [ ]:
# custom function to be executed in a child process
def task(arg1, arg2, arg3):
    # report a message
    print(f"From another process {arg1}, {arg2}, {arg3}", flush=True)
    # return a value
    return arg1 + arg2+ arg3

# protect the entry point 
if __name__ == "__main__":
    # create a multiprocessing pool
    with Pool() as pool:
        # prepare task arguments
        args = [(i, i*2, i*3) for i in range(10)]
        # issue multiple tasks to the pool
        async_result = pool.starmap_async(task, args)
        # process return values
        for result in async_result.get():
            print(result)

## Execute Tasks in Parallel and Be Responsive
**1. Limitations of the `map()` method.**

A problem with the `map()` method to the `Pool` is that it traverses the provided iterable and issues all tasks to the multiprocessing pool immediately.

This can be a problem if the iterable contains many hundreds or thousads of items. This is because the multiprocessing pool will then have hundreds, thousands, or millions of tasks sitting idly waiting to execute, using large amounts of main memory unnecessarily.

Alternative, `imap()` method which is a lazy version of `map()`:
* Argument items are yileded from the iterable as workers become available, rather than of all at once.
* Return values are yielded in order as they are completed, rather than after all tasks are completed.

A limitation of `imap()` method is that it yields return value results in the order that the tasks were issued.

The `imap_unordered()` addresses this limitation. Return values are yielded in the order that tasks are completed, not the order that the tasks were issued to the multiprocessing pool.

**2. Issue tasks as needed and process results as they become available.**

In [ ]:
# custom function to be executed in a child process
def task(arg):
    # block for a random, fraction of a second
    sleep(random())
    # report a message
    print(f"From another process {arg}", flush=True)
    # return a value
    return arg * 2

# protect the entry point 
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool(4) as pool:
        # issue multiple tasks and process return values
        for result in pool.imap(task, range(10)):
            print(result)

**3. Process results from issued tasks in the order they are completed.**

The iterable will yield return values as tasks are completed, in the order that tasks were completed, not the order they were issued.

`imap_unordered()` capabilities:
* Issue multiple tasks to the multiprocessing pool, one-by-one.
* Return an iterable over return values, results are yielded as tasks complete.
* Supports a single argument to the target function.
* Block until each task is completed in the order they are completed.
* Allow tasks to be grouped and executed in batches by workers.

In [ ]:
# custom function to be executed in a child process
def task(arg):
    # block for a random fraction of a second
    sleep(random())
    # report a message
    print(f"From another process {arg}", flush=True)
    # return a value
    return arg * 2

# protect the entry point
if __name__ == "__main__":
    # create a multiprocessing pool
    with Pool(4) as pool:
        # issue multiple tasks and processes return values
        for rs in pool.imap_unordered(task, range(10)):
            print(rs)

## Callbacks and `AsyncResults` for Async Tasks
**1. Add callback functions to asynchronous tasks.**

A callback is a function that is first registered and then called automatically by the multiprocessing pool on some event.

Only supported in the multiprocessing pool when issuing tasks asynchronously.

Called in two situations:
* *Result*: With the results of a task when the task finishes successfully. Specified via `callback=`.
* *Error*: When an exception or error is raised in a task and is not handled. Specified via `error_callback=`.

In [1]:
# result callback function
def result_callback(return_value):
    # report a message
    print(f"Callback got: {return_value}", flush=True)
    
# custom function to be executed in a child process
def task(identifier):
    # generate a value
    value = random()
    # report a message
    print(f"Task {identifier} executing with {value}", flush=True)
    # block for a moment
    sleep(value)
    # return the generated value
    return value

# protect the entry point
if __name__ == "__main__":
    # create and configure the multiprocessing pool
    with Pool() as pool:
        # issue tasks to the multiprocessing pool
        result = pool.apply_async(task, args=(0,), callback=result_callback)
        # close the multiprocessing pool
        pool.close()
        # wait for all tasks to complete
        pool.join()

**2. Wait for and get results from asynchronous tasks.**

An `AsyncResult` object is returned when issuing tasks to `Pool` the multiprocessing pool asynchonously. We can use it to interact with the task in two ways:
* Get the result of the issued task or tasks. Use `get()` method. If the issued tasks have not yet completed, then `get()` will block until the tasks are finished.
```python
try:
    value = async_result.get(timeout=10)
except TimeoutError as e:
    ...
except Exception as e:
    ...
```
* Wait for the issued task or all issued tasks to complete. Use `wait()` method. Will block until all issued tasks are completed. Completed means that the task or tasks completed successfully, or failed with an unhandled error or exception. When using a timeout, the `wait()` method does not give an indication that it returned because tasks completed or because the timeout elapsed. Therefore, we can check if the tasks completed via the `ready()` method.
```python
async_result.wait(timeout=10)
if async_result.ready():
    print("All Done")
else:
    print("Not Done Yet")
```


**3. Check the status of asynchronous tasks.**
* Check if the task or tasks have been completed. Use `ready()`. `True` if the tasks have been completed, successfully or otherwise. `False` if the tasks are still running.
```python
if async_result.ready():
    print("Tasks are done")
else:
    print("Tasks are not done")
```
* Check if all tasks were successful. Use `successful()`. Issued tasks are successful if no tasks raised an exception. If at least one issued task raised an exception return `False`.
```python
if async_result.ready():
    try:
        if async_result.successful():
            print("Successful")
        else:
            print("Unsuccessful")
    except ValueError as e:
        print("Tasks still running")
```

In [ ]:
# custom function to be executed in a child process
def task():
    # loop a few times to simulate a slow task
    for i in range(10):
        # generate a random value between 0 and 1
        value = random()
        # block for a fraction of a second
        sleep(value)
        # report a message
        print(f">{i} got {value}", flush=True)
        
# protect the entry point
if __name__ == "__main__":
    # create the multiprocessing pool
    with Pool() as pool:
        # issue a task asynchronously
        async_result = pool.apply_async(task)
        # wait until the task is complete
        while not async_result.ready():
            # report a message
            print("Main process waiting ...")
            # block for a moment
            async_result.wait(timeout=1)
        # report if the task was successful
        if async_result.successful():
            print("Task was successful.")

Busy-wait loop and is a helpful pattern in concurrency programming. It allows the caller to do other things and give up waiting if it chooses. The task loops, generating random numbers and reporting progress along the way.

## Case Study: Parallel Primality Testing
* If the number is less than 2, it is not prime.
* If the number is 2, prime.
* If the number can be divided by 2 with no reminder, not prime.
* If the number has a divisor between 3 and `sqrt(n)`, not prime. Side note: Odd numbers between 3 and the target number minus one. Skip the others because they are divisible by 2.
* Otherwise, prime.

In [ ]:
# returns True if prime, False otherwise
def is_prime(number):
    # 1 is a special case of not prime
    if number <= 1:
        return False
    # 2 is a special case of a prime
    if number == 2:
        return True
    # check if the number divides by 2 with no remainder
    if number % 2 == 0:
        return False
    # limit divisors to sqrt(n) + 1 so range will reach it
    limit = floor(sqrt(number)) + 1
    # check all odd numbers in range
    for i in range(3, limit, 2):
        # check if number is divisible and is not a prime
        if number % i == 0:
            # number is divisible and is not a prime
            return False
    # number is probably prime
    return True

**Slow**

In [ ]:
# check if a series of numbers are prime or not
def check_numbers_are_prime(numbers):
    # check each number is turn
    for number in numbers:
        if is_prime(number):
            print(f"{number} is prime")
            
# protect the entry point
if __name__ == "__main__":
    # define some numbers to check
    NUMS = [17977, 10619863, 106198, 6620830889, 80630964769, 228204732751, 1171432692373, 1398341745571,
            10963707205259, 15285151248481, 99999199999, 304250263527209, 30425026352720, 10657331232548839,
            10657331232548830, 44560482149, 1746860020068409]
    # check whether each number is a prime
    check_numbers_are_prime(NUMS)

**Fast - in parallel**

In [ ]:
# check if a series of numbers are prime or not
def check_numbers_are_prime(numbers):
    # create a multiprocessing pool
    with Pool() as pool:
        # issue the tasks
        results = pool.imap(is_prime, numbers)
        # report the results as completed in order
        for number, isprime in zip(numbers, results):
            if isprime:
                print(f"{number} is prime")
            
# protect the entry point
if __name__ == "__main__":
    # define some numbers to check
    NUMS = [17977, 10619863, 106198, 6620830889, 80630964769, 228204732751, 1171432692373, 1398341745571,
            10963707205259, 15285151248481, 99999199999, 304250263527209, 30425026352720, 10657331232548839,
            10657331232548830, 44560482149, 1746860020068409]
    # check whether each number is a prime
    check_numbers_are_prime(NUMS)